# Case 7 · Where should the Gold tier start? (Part 2)
**BUS-G 492 · Causal decisions with AI**

In [ ]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case7_rdd_gold/data/"
for candidate in ["data/", "cases/case7_rdd_gold/data/", "../../../cases/case7_rdd_gold/data/", "../../../../cases/case7_rdd_gold/data/"]:
    if os.path.exists(candidate + "customers.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

## Stage 5 · Something you didn't know

> **From:** Ben Kowalczyk, Loyalty Operations
> **Re:** how Gold works, exactly
>
> Gold isn't something customers apply for. It's **automatic**: anyone whose spending in a calendar year is at least **$1,000.00** gets Gold for the next year. No exceptions. (Rules: `gold_rules.csv`.)
>
> One quirk: every December, some customers who are just short of $1,000 make a **small extra purchase** to qualify.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. Run this cell to load the Part 1 data again, then the next cell for the new files.

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "customers.csv")
print(df.shape)
df.head()

In [ ]:
rules = pd.read_csv(DATA + "gold_rules.csv"); print(rules.rule[0])

## Your tasks

Use AI again, but this time **tell it the whole story**: how the data were generated, and what the new information is. Each step ends with a table: fill it in (double-click the cell to type) before moving on.

### Step 1 · Picture it

Keep customers with 2025 spending between $900 and $1,100. Put them in **$25 bins** of 2025 spending and plot average 2026 spending per bin. Fill in a few bins:

| 2025 spending bin | Average 2026 spending |
|---|---|
| $950–975 | |
| $975–1,000 | |
| $1,000–1,025 | |
| $1,025–1,050 | |

Is there a jump at $1,000?

In [ ]:
near = df[(df.spend_2025 >= 900) & (df.spend_2025 < 1100)].copy()
near["bin"] = (near.spend_2025 // 25 * 25).astype(int)
# Your code here (hint: near.groupby("bin").spend_2026.mean().plot(marker="o"))


### Step 2 · Estimate the jump (regression discontinuity) → ETT at the cutoff

Let x = 2025 spending − $1,000. Within a **bandwidth** around the cutoff, regress 2026 spending on `gold_2026`, x, and `gold_2026 × x` (a separate line on each side). The coefficient on `gold_2026` is the jump: the **effect on the treated (ETT) at the cutoff**.

| Bandwidth | Jump ($) | Standard error |
|---|---|---|
| ± $100 | | |
| ± $200 | | |
| ± $300 | | |

In [ ]:
df["x"] = df.spend_2025 - 1000
for bw in [100, 200, 300]:
    d = df[df.x.abs() <= bw]
    # Your code here: m = smf.ols("spend_2026 ~ gold_2026 * x", d).fit(); print the gold_2026 coefficient and standard error
    pass


### Step 3 · Check for gaming (bunching)

Some customers just below $1,000 make a small extra purchase in December to qualify. Count customers per **$10 bin** of 2025 spending:

| Bin | $960 | $970 | $980 | $990 | $1,000 | $1,010 | $1,020 | $1,030 |
|---|---|---|---|---|---|---|---|---|
| Customers | | | | | | | | |

Is there a pile-up just above $1,000? Re-estimate Step 2 **leaving out** customers within $25 of the cutoff (a "donut"):

| Bandwidth | Donut jump ($) |
|---|---|
| ± $200 | |
| ± $300 | |

In [ ]:
# Your code here (hint: (df.spend_2025 // 10 * 10).value_counts().sort_index().loc[960:1030])


### Step 4 · Into dollars → choose T

Express the jump as a % of spending just below the cutoff (about $900). Gold perks cost **$120** per member, and margin is **35%**:

| 2026 spending | Value per Gold member = 0.35 × (jump %) × spending − $120 |
|---|---|
| $1,000 | |
| $2,000 | |
| $3,000 | |
| $4,000 | |
| $6,000 | |

Choose T. What must you **assume** to use the effect measured at $1,000 for customers who spend $3,000 or more? Predicted profit = Σ value over customers with 2026 spending ≥ T, × 10 for all customers.

In [ ]:
# Your code here


### Decide again

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
GOLD_THRESHOLD = None      # minimum 2026 spending for Gold in 2027 ($); 999999 = scrap the tier  (e.g. 1000)
PREDICTED_PROFIT = None    # in dollars, as a number, e.g. 150000
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""

# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("GOLD_THRESHOLD", GOLD_THRESHOLD), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              7")
print("round:             2  (Part II)")
print(f"decision:          {GOLD_THRESHOLD}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

## Submit (round 2)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Predicted profit:** a number such as `150000` (`$150,000` and `150K` also work, but not words). To scrap the Gold tier, enter `999999` as the decision.
- **Case:** `7`.  **Round:** `2` (Part II).
- **Submissions are final.** You can't change an answer after you submit, so check the printed values first. If your team submits more than once for the same round, only the **first** submission counts.
- Use the **same team name** in every round.

## Reflection (submit)

1. Your Part 1 analysis may have fit the data very well. Why was it still the wrong tool for this decision?
2. In one sentence: what was the difference between **predicting** the outcome and **deciding** what to do?
3. Name one other business decision where the same trap could appear.